In [ ]:
# Example 1: Public API surface — verify primary exports are present.
import nvitop

expected_names = [
    'Device',
    'HostProcess',
    'GpuProcess',
    'libnvml',
    'NA',
    'host',
]

present = {name: hasattr(nvitop, name) for name in expected_names}
for name, ok in present.items():
    print(f"  nvitop.{name}: {'present' if ok else 'MISSING'}")

assert all(present.values()), f"Missing public API names: {[n for n, ok in present.items() if not ok]}"


In [ ]:
# Example 2: Device class — query class-level GPU counts via NVML.
# Runs without requiring a present GPU; if NVML init fails, we surface it cleanly.
from nvitop import Device

try:
    count = Device.count()
    print(f"Device.count() = {count}")
    all_devices = Device.all()
    print(f"Device.all() returned {len(all_devices)} device(s)")
    assert isinstance(count, int)
    assert isinstance(all_devices, list)
    assert len(all_devices) == count
except Exception as e:
    # NVML may not be available in this environment; the class itself must still import.
    print(f"NVML unavailable ({type(e).__name__}: {str(e)[:120]}); class import OK")
    assert Device is not None


In [ ]:
# Example 3: host module — system-wide CPU + memory stats via psutil wrapper.
from nvitop import host

cpu_percent = host.cpu_percent(interval=None)
virt_mem = host.virtual_memory()
swap_mem = host.swap_memory()

print(f"host.cpu_percent = {cpu_percent}")
print(f"host.virtual_memory().total = {virt_mem.total}")
print(f"host.swap_memory().total = {swap_mem.total}")

assert isinstance(cpu_percent, (int, float))
assert virt_mem.total > 0
assert hasattr(swap_mem, 'total')


In [ ]:
# Example 4: HostProcess — wrap the current process and pull a few attributes.
import os

from nvitop import HostProcess

proc = HostProcess(os.getpid())
name = proc.name()
cmdline = proc.cmdline()
running = proc.is_running()

print(f"HostProcess pid = {proc.pid}")
print(f"HostProcess.name() = {name}")
print(f"HostProcess.cmdline() has {len(cmdline)} part(s)")
print(f"HostProcess.is_running() = {running}")

assert proc.pid == os.getpid()
assert isinstance(name, str) and name
assert running is True


In [ ]:
# Example 5: NA sentinel + utility helpers — bytes / time formatters.
from nvitop import NA, bytes2human, timedelta2human

# NA behaves like a string-ish sentinel for unavailable values.
print(f"NA = {NA!r}")
print(f"str(NA) = {str(NA)!r}")

human_bytes = bytes2human(1536 * 1024 * 1024)  # ~1.5 GiB
human_time = timedelta2human(3725)  # 1h 2m 5s
print(f"bytes2human(1.5 GiB) = {human_bytes!r}")
print(f"timedelta2human(3725s) = {human_time!r}")

# NA round-trips through str; formatters return non-empty strings.
assert str(NA)
assert isinstance(human_bytes, str) and human_bytes
assert isinstance(human_time, str) and human_time
